In [1]:
# Q9 - Economic context

# Question: does the economic climate change the yes rate?

# The five indicators are national figures for the time of the call (decision 5), not customer attributes: employment variation
# rate and number employed are quarterly, consumer price index and consumer confidence index are monthly, and the euribor 3 month
# rate is daily. They describe the same periods, so they are expected to move together. The data has no year column, so each value
# stands for a period, not for a customer. Each group is compared with the overall yes rate (the Q1 baseline) and shown with its
# customer count. Groups under 100 customers are flagged and not interpreted.

In [2]:
import pandas as pd

CLEAN_PATH = "../../data/bank_marketing_clean.parquet"
clean_df = pd.read_parquet(CLEAN_PATH)

clean_df.shape

(41176, 22)

In [3]:
# Q9, step 1: yes rate at each level of each economic indicator
subscribed = clean_df["subscribed_term_deposit"] == "yes"
baseline_rate = subscribed.mean() * 100

# Quarterly indicators have only 10-11 distinct values, so each value is its own group;
# the others are cut into five equal-sized bands
indicators = {
    "employment_variation_rate": "each value",
    "number_employed": "each value",
    "euribor_3_month_rate": "five bands",
    "consumer_price_index": "five bands",
    "consumer_confidence_index": "five bands",
}

def summarise(column, how):
    values = clean_df[column]
    groups = pd.qcut(values, q=5, duplicates="drop") if how == "five bands" else values
    summary = (
        pd.DataFrame({"group": groups, "subscribed": subscribed})
        .groupby("group", observed=True)["subscribed"]
        .agg(customers="count", subscribers="sum")
    )
    summary["yes_rate_pct"] = (summary["subscribers"] / summary["customers"] * 100).round(2)
    summary["vs_baseline_points"] = (summary["yes_rate_pct"] - baseline_rate).round(2)
    summary["share_of_customers_pct"] = (summary["customers"] / len(clean_df) * 100).round(2)
    summary["share_of_subscribers_pct"] = (summary["subscribers"] / subscribed.sum() * 100).round(2)
    summary["small_group"] = summary["customers"] < 100
    summary.index = summary.index.astype(str)
    print(f"{column}: {len(summary)} groups, customers {summary['customers'].sum()} "
          f"(should be {len(clean_df)}), subscribers {summary['subscribers'].sum()} (should be {subscribed.sum()})")
    return summary

summaries = [summarise(column, how) for column, how in indicators.items()]

print(f"Baseline yes rate: {baseline_rate:.2f}%")
print()
print("How closely the indicators move together (1 = identical, 0 = unrelated):")
print(clean_df[list(indicators)].corr().round(2))
print()
pd.concat(summaries, keys=list(indicators), names=["indicator", "group"])

employment_variation_rate: 10 groups, customers 41176 (should be 41176), subscribers 4639 (should be 4639)
number_employed: 11 groups, customers 41176 (should be 41176), subscribers 4639 (should be 4639)
euribor_3_month_rate: 5 groups, customers 41176 (should be 41176), subscribers 4639 (should be 4639)
consumer_price_index: 5 groups, customers 41176 (should be 41176), subscribers 4639 (should be 4639)
consumer_confidence_index: 5 groups, customers 41176 (should be 41176), subscribers 4639 (should be 4639)
Baseline yes rate: 11.27%

How closely the indicators move together (1 = identical, 0 = unrelated):
                           employment_variation_rate  number_employed  \
employment_variation_rate                       1.00             0.91   
number_employed                                 0.91             1.00   
euribor_3_month_rate                            0.97             0.95   
consumer_price_index                            0.78             0.52   
consumer_confidence_ind

customers  \
indicator                 group                                     
employment_variation_rate -3.4                               1070   
                          -3.0                                172   
                          -2.9                               1662   
                          -1.8                               9182   
                          -1.7                                773   
                          -1.1                                635   
                          -0.2                                 10   
                          -0.1                               3682   
                          1.1                                7762   
                          1.4                               16228   
number_employed           4963.6                              635   
                          4991.6                              773   
                          5008.7                              650   
                          5017.5                             1070   
                          5023.5                              172   
                          5076.2                             1662   
                          5099.1                             8532   
                          5176.3                               10   
                          5191.0                             7762   
                          5195.8                             3682   
                          5228.1                            16228   
euribor_3_month_rate      (0.633, 1.299]                     8634   
                          (1.299, 4.191]                     8427   
                          (4.191, 4.864]                     8445   
                          (4.864, 4.962]                     8496   
                          (4.962, 5.045]                     7174   
consumer_price_index      (92.19999999999999, 92.893]        8274   
                          (92.893, 93.444]                  12224   
                          (93.444, 93.918]                   7134   
                          (93.918, 93.994]                   7762   
                          (93.994, 94.767]                   5782   
consumer_confidence_index (-50.800999999999995, -46.2]       8864   
                          (-46.2, -42.0]                    10306   
                          (-42.0, -40.0]                     5679   
                          (-40.0, -36.4]                     8527   
                          (-36.4, -26.9]                     7800   

                                                        subscribers  \
indicator                 group                                       
employment_variation_rate -3.4                                  454   
                          -3.0                                   88   
                          -2.9                                  593   
                          -1.8                                 1461   
                          -1.7                                  403   
                          -1.1                                  301   
                          -0.2                                    1   
                          -0.1                                  232   
                          1.1                                   240   
                          1.4                                   866   
number_employed           4963.6                                301   
                          4991.6                                403   
                          5008.7                                369   
                          5017.5                                454   
                          5023.5                                 88   
                          5076.2                                593   
                          5099.1                               1092   
                          5176.3                                  1   
             

In [4]:
# Q9, step 2: periods in date order, and the months inside each period
month_order = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
strong_months = ["mar", "sep", "oct", "dec"]

number_employed = clean_df["number_employed"]
period = (number_employed != number_employed.shift()).cumsum()   # a new period starts whenever the value changes

frame = pd.DataFrame({
    "period": period,
    "number_employed": number_employed,
    "month": clean_df["contact_month"],
    "month_type": clean_df["contact_month"].isin(strong_months).map({True: "strong months", False: "other months"}),
    "subscribed": subscribed,
    "row": clean_df.index,
})

# Reconcile against the earlier tables
expected_by_value = {4963.6: 635, 4991.6: 773, 5008.7: 650, 5017.5: 1070, 5023.5: 172, 5076.2: 1662,
                     5099.1: 8532, 5176.3: 10, 5191.0: 7762, 5195.8: 3682, 5228.1: 16228}
actual_by_value = {round(value, 1): count for value, count in frame.groupby("number_employed").size().items()}
print(f"Customers: {len(frame)} (should be {len(clean_df)}), subscribers: {frame['subscribed'].sum()} (should be {subscribed.sum()})")
print(f"Customers per number_employed value match the last table: {actual_by_value == expected_by_value}")
print(f"Customers per month match Question 5: "
      f"{frame.groupby('month').size().reindex(month_order).tolist() == [546, 2631, 13767, 5318, 7169, 6176, 570, 717, 4100, 182]}")
print(f"Periods found: {frame['period'].nunique()} (11 if every value forms one block)")
print()

# 1. The periods, in date order
periods = frame.groupby("period").agg(
    number_employed=("number_employed", "first"),
    first_row=("row", "min"),
    last_row=("row", "max"),
    customers=("subscribed", "count"),
    subscribers=("subscribed", "sum"),
    months=("month", lambda months: " ".join(dict.fromkeys(months))),
)
periods["yes_rate_pct"] = (periods["subscribers"] / periods["customers"] * 100).round(2)
periods["small_group"] = periods["customers"] < 100
print("Periods in date order:")
print(periods)
print()

# 2. Yes rate for each month inside each period (blank where under 100 customers)
by_month = frame.groupby(["period", "month"], sort=False)["subscribed"].agg(customers="count", subscribers="sum")
by_month["yes_rate_pct"] = (by_month["subscribers"] / by_month["customers"] * 100).where(by_month["customers"] >= 100).round(2)
print("Yes rate by period and month (blank where under 100 customers):")
print(by_month)
print()

# 3. Periods that contain both a strong month and another month
by_type = frame.groupby(["period", "month_type"])["subscribed"].agg(customers="count", subscribers="sum")
by_type["yes_rate_pct"] = (by_type["subscribers"] / by_type["customers"] * 100).where(by_type["customers"] >= 100).round(2)
mixed_periods = frame.groupby("period")["month_type"].nunique()
mixed_periods = mixed_periods[mixed_periods == 2].index
print("Periods containing both strong months and other months:")
by_type.loc[mixed_periods]

Customers: 41176 (should be 41176), subscribers: 4639 (should be 4639)
Customers per number_employed value match the last table: True
Customers per month match Question 5: True
Periods found: 11 (11 if every value forms one block)

Periods in date order:
        number_employed  first_row  last_row  customers  subscribers  \
period                                                                 
1                5191.0          0      7761       7762          240   
2                5228.1       7762     23989      16228          866   
3                5195.8      23990     27671       3682          232   
4                5176.3      27672     27681         10            1   
5                5099.1      27682     36213       8532         1092   
6                5076.2      36214     37875       1662          593   
7                5017.5      37876     38945       1070          454   
8                5023.5      38946     39117        172           88   
9                5008.7  

customers  subscribers  yes_rate_pct
period month_type                                         
3      other months        3615          190          5.26
       strong months         67           42           NaN
5      other months        8250          966         11.71
       strong months        282          126         44.68
7      other months         357          168         47.06
       strong months        713          286         40.11
9      other months         386          219         56.74
       strong months        264          150         56.82
11     other months         128           58         45.31
       strong months        507          243         47.93

In [5]:
# Q9, step 3: do the main findings hold within each inferred year?
year_by_period = {1: 2008, 2: 2008, 3: 2008, 4: 2008, 5: 2009, 6: 2009, 7: 2009, 8: 2009,
                  9: 2010, 10: 2010, 11: 2010}
year = period.map(year_by_period)   # inferred from the row order, not a column in the data

age_edges = [0, 25, 35, 45, 55, 65, 120]
age_labels = ["under 25", "25-34", "35-44", "45-54", "55-64", "65 and over"]
age_group = pd.cut(clean_df["age"], bins=age_edges, labels=age_labels, right=False)

print(f"Customers by year: {year.value_counts().sort_index().to_dict()} "
      f"(should be {{2008: 27682, 2009: 11436, 2010: 2058}})")
print(f"Subscribers by year: {subscribed.groupby(year).sum().to_dict()} "
      f"(should be {{2008: 1339, 2009: 2227, 2010: 1073}})")
print()

def by_year_table(title, values, order):
    grouped = (pd.DataFrame({"value": values.astype(str), "year": year, "subscribed": subscribed})
               .groupby(["value", "year"])["subscribed"].agg(customers="count", subscribers="sum"))
    customers = grouped["customers"].unstack("year").reindex(order).fillna(0).astype(int)
    subscribers = grouped["subscribers"].unstack("year").reindex(order).fillna(0).astype(int)
    rates = (subscribers / customers * 100).where(customers >= 100).round(1)
    result = pd.concat({"customers": customers, "yes_rate_pct": rates}, axis=1)
    result[("yes_rate_pct", "all years")] = (subscribers.sum(axis=1) / customers.sum(axis=1) * 100).round(1)
    print(f"{title} (rates blank where under 100 customers):")
    print(f"  customers by year: {customers.sum().to_dict()}, total {customers.values.sum()} (should be 41176)")
    print(result)
    print()

by_year_table("Contact type", clean_df["contact_type"], ["cellular", "telephone"])
by_year_table("Previous campaign outcome", clean_df["previous_campaign_outcome"],
              ["nonexistent", "failure", "success"])
by_year_table("Age group", age_group, age_labels)

Customers by year: {2008: 27682, 2009: 11436, 2010: 2058} (should be {2008: 27682, 2009: 11436, 2010: 2058})
Subscribers by year: {2008: 1339, 2009: 2227, 2010: 1073} (should be {2008: 1339, 2009: 2227, 2010: 1073})

Contact type (rates blank where under 100 customers):
  customers by year: {2008: 27682, 2009: 11436, 2010: 2058}, total 41176 (should be 41176)
          customers              yes_rate_pct                      
year           2008   2009  2010         2008  2009  2010 all years
value                                                              
cellular      13970  10495  1670          5.7  19.9  57.7      14.7
telephone     13712    941   388          3.9  14.7  28.4       5.2

Previous campaign outcome (rates blank where under 100 customers):
  customers by year: {2008: 27682, 2009: 11436, 2010: 2058}, total 41176 (should be 41176)
            customers            yes_rate_pct                      
year             2008  2009 2010         2008  2009  2010 all years
val

In [6]:
# Q9, step 4: do the Question 3, 4 and 6 findings hold within each inferred year?
contacts = clean_df["contacts_this_campaign"]
call_group = pd.cut(contacts, bins=[1, 2, 3, 4, 6, 11, 1000],
                    labels=["1", "2", "3", "4-5", "6-10", "11 and over"], right=False)

by_year_table("Job", clean_df["job"],
              ["student", "retired", "unemployed", "admin.", "management", "unknown",
               "technician", "self-employed", "housemaid", "entrepreneur", "services", "blue-collar"])
by_year_table("Education", clean_df["education"],
              ["illiterate", "basic.4y", "basic.6y", "basic.9y", "high.school",
               "professional.course", "university.degree", "unknown"])
by_year_table("Credit default status", clean_df["credit_default"], ["no", "unknown", "yes"])
by_year_table("Calls received in this campaign", call_group,
              ["1", "2", "3", "4-5", "6-10", "11 and over"])

Job (rates blank where under 100 customers):
  customers by year: {2008: 27682, 2009: 11436, 2010: 2058}, total 41176 (should be 41176)
              customers            yes_rate_pct                      
year               2008  2009 2010         2008  2009  2010 all years
value                                                                
student             193   473  209          7.3  32.1  52.2      31.4
retired             795   649  274          5.9  36.5  54.7      25.3
unemployed          662   266   86          3.6  27.4   NaN      14.2
admin.             6827  2951  641          4.9  22.8  53.4      13.0
management         2015   789  120          5.4  20.0  51.7      11.2
unknown             248    64   18          5.2   NaN   NaN      11.2
technician         4870  1552  317          4.7  21.5  52.7      10.8
self-employed       992   376   53          4.8  20.5   NaN      10.5
housemaid           824   181   55          3.3  26.5   NaN      10.0
entrepreneur       1066 